# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoye-J/FlyRank--MachineLearning/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import os
import pandas as pd
import numpy as np
import duckdb

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")

FACT_MAR    = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"
DIM_CLIENTS = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"

print("Token loaded:", os.environ["HF_TOKEN"][:7] + "..." + os.environ["HF_TOKEN"][-4:])
print("Connected to warehouse.")

Token loaded: hf_QLrn...EGIi
Connected to warehouse.


## 1. Unit of analysis + time window


One row = one page per day, a single content item's daily performance for a single client on a single report date.

Grain key:(report_date, client_hash_id, content_hash_id)

Time window:month=2026-03 (March 2026), a mid-panel month.

Why March, not June: the 'fact_content_daily_performance_sample.parquet' file is exactly the final month (June 2026), not a random sample. For a past to future label, June is the natural outcome window. Developing label logic there would put me inside my own test window. March is safely mid-panel: ~14 months of history behind it and ~3 months ahead of it.

Tables I will use:
- 'fact_content_daily_performance/month=2026-03/*.parquet'  daily performance facts.
- 'dim_content.parquet' static content attributes (content_type, main_intent, word_count).
- 'dim_clients.parquet' client history depth (gsc_data_start, ga4_data_start) for context and the imbalance check.

What I predict/rank: a priority score per page for intent-content review. The underlying observed signal is a future-window decline in GSC impressions/clicks (April–May 2026).

One thing I deliberately exclude: all rows from month=2026-06 (the sealed test month). I also exclude every 'ga4_*' value on rows where 'ga4_data_available = FALSE' zeros there are fill, not measured engagement.

In [2]:
# Peek at the March slice to confirm columns and grain
display(con.execute(f"SELECT * FROM '{FACT_MAR}' LIMIT 3").df())

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

Every field I may touch goes in exactly one bucket.

FEATURE
- 'gsc_impressions' (prior window) measured before the decision.
- 'gsc_clicks' (prior window) measured before the decision.
- 'gsc_avg_position' (prior window, excluding 0) measured before the decision.
- 'sessions_ai' (prior window) AI-search intent signal, measured before.
- 'content_type', 'main_intent', 'word_count' (from 'dim_content') static page attributes.

LABEL / PROXY (not a feature)
- 'is_declining__future_window' constructed from a future window of GSC performance (April–May 2026).
- 'trend_direction' starter CSV only (warehouse does not ship it; it is the label source).
- 'trend_pct' same; the exact number the label is derived from.

CONTEXT (grouping, joining, splitting; never learned from)
- 'client_hash_id' grouped train/test split.
- 'content_hash_id' join fact and dim_content.
- 'report_date' time-window slicing.
- 'client_has_gsc', 'client_has_ga4' context flags.
- 'has_gsc_access', 'has_ga4_access' (dim_clients) context.

EXCLUDED (private / product-decision / future info)
- 'ga4_*' and 'gsc_*' when '*_data_available = FALSE' zeros are fill, not measurement.
- 'is_published', 'is_deleted' (dim_content) product-decision flags.
- 'provider_used', 'model_used' (dim_content) internal product metadata.
- All rows from 'month=2026-06'' sealed test month.

In [3]:
field_classification = {
    "feature": [
        "gsc_impressions__prior_window",
        "gsc_clicks__prior_window",
        "gsc_avg_position__prior_window",
        "sessions_ai__prior_window",
        "content_type",
        "main_intent",
        "word_count",
    ],
    "label_or_proxy": [
        "is_declining__future_window",
        "trend_direction (starter CSV only)",
        "trend_pct (starter CSV only)",
    ],
    "context": [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "client_has_gsc",
        "client_has_ga4",
        "has_gsc_access",
        "has_ga4_access",
    ],
    "excluded": [
        "ga4_* where ga4_data_available = FALSE",
        "gsc_* where gsc_data_available = FALSE",
        "is_published, is_deleted",
        "provider_used, model_used",
        "all rows from month=2026-06",
    ],
}

for bucket, fields in field_classification.items():
    print(f"\n{bucket.upper()}:")
    for f in fields:
        print(f"  - {f}")


FEATURE:
  - gsc_impressions__prior_window
  - gsc_clicks__prior_window
  - gsc_avg_position__prior_window
  - sessions_ai__prior_window
  - content_type
  - main_intent
  - word_count

LABEL_OR_PROXY:
  - is_declining__future_window
  - trend_direction (starter CSV only)
  - trend_pct (starter CSV only)

CONTEXT:
  - client_hash_id
  - content_hash_id
  - report_date
  - client_has_gsc
  - client_has_ga4
  - has_gsc_access
  - has_ga4_access

EXCLUDED:
  - ga4_* where ga4_data_available = FALSE
  - gsc_* where gsc_data_available = FALSE
  - is_published, is_deleted
  - provider_used, model_used
  - all rows from month=2026-06


## 3. Verify it with queries (grain, counts, missing values, windows)


Three queries on month=2026-03. Every claim above gets a query.

1. Grain: 'GROUP BY report_date, client_hash_id, content_hash_id HAVING COUNT(*) > 1' should return zero rows.
2. Counts + date span: total rows, distinct days, distinct clients, distinct content, min/max date.
3. Availability: count rows where 'ga4_data_available IS TRUE' vs total, proving the zeros-are-fill trap.

In [4]:
print("1) GRAIN: one row = one (report_date, client_hash_id, content_hash_id)")
q1 = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM '{FACT_MAR}'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""
grain_check = con.execute(q1).df()
if grain_check.empty:
    print("   Zero rows back — the grain HOLDS. One row = one page-day.")
else:
    display(grain_check)
print()

print("2) COUNTS + DATE SPAN of my March 2026 slice")
q2 = f"""
SELECT
    COUNT(*)                        AS total_rows,
    COUNT(DISTINCT report_date)     AS n_days,
    COUNT(DISTINCT client_hash_id)  AS n_clients,
    COUNT(DISTINCT content_hash_id) AS n_content,
    MIN(report_date)                AS min_date,
    MAX(report_date)                AS max_date
FROM '{FACT_MAR}'
"""
display(con.execute(q2).df())
print()

print("3) AVAILABILITY: GSC vs GA4 flag survivors")
q3 = f"""
SELECT
    COUNT(*)                                                AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)      AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)      AS ga4_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE)  AS ga4_unavailable_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_ga4_available
FROM '{FACT_MAR}'
"""
display(con.execute(q3).df())

1) GRAIN: one row = one (report_date, client_hash_id, content_hash_id)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Zero rows back — the grain HOLDS. One row = one page-day.

2) COUNTS + DATE SPAN of my March 2026 slice


,total_rows,n_days,n_clients,n_content,min_date,max_date
0,9841378,31,55,331437,2026-03-01,2026-03-31



3) AVAILABILITY: GSC vs GA4 flag survivors


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,ga4_unavailable_rows,pct_ga4_available
0,9841378,3611061,413966,9427412,4.2


## 4. Data limits


Main limitation: unbalanced client history. Each client's GSC and GA4 data start on different dates. In March 2026, some clients have ~14 months of history behind them and others have very little. A "prior 30 days" feature is not the same quantity for a new client as for a mature one.

Other limits I accept:
- Rows before 'ga4_data_start' have 'ga4_data_available = FALSE' and GA4 columns zero-filled. Zeros there are NOT "no engagement."
- 'fact_content_query_90d.parquet' has a fixed 90-day window that overlaps the daily panel's recent months, I cannot use its columns as-is for a past to future label without checking window alignment.
- A single month cannot represent seasonality.
- The starter CSV and the warehouse do not ship the same label: the warehouse does not have 'trend_direction' / 'trend_pct'. I construct my own future-window decline label from GSC columns; the CSV's label was used only for practice.

How I will handle it in modeling: use per-client windows relative to 'dim_clients.gsc_data_start', not one global calendar window.

In [5]:
q_summary = f"""
SELECT
    COUNT(*) AS n_clients,
    MIN(gsc_data_start) AS earliest_gsc_start,
    MAX(gsc_data_start) AS latest_gsc_start,
    COUNT(*) FILTER (WHERE gsc_data_start IS NULL) AS clients_no_gsc_start
FROM '{DIM_CLIENTS}'
"""
print("Client history depth summary:")
display(con.execute(q_summary).df())
print()

q_buckets = f"""
SELECT
    DATE_TRUNC('month', gsc_data_start) AS gsc_start_month,
    COUNT(*) AS n_clients
FROM '{DIM_CLIENTS}'
GROUP BY 1
ORDER BY 1
"""
print("Clients by GSC history start month (this is the imbalance):")
display(con.execute(q_buckets).df())

Client history depth summary:


,n_clients,earliest_gsc_start,latest_gsc_start,clients_no_gsc_start
0,104,2025-01-27,2026-06-02,37



Clients by GSC history start month (this is the imbalance):


,gsc_start_month,n_clients
0,NaT,37
1,2025-01-01,2
2,2025-02-01,1
3,2025-03-01,1
4,2025-06-01,5
5,2025-07-01,7
6,2025-09-01,8
7,2025-10-01,5
8,2025-11-01,10
9,2025-12-01,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.